In [ ]:
# 1. 패키지 설치 (터미널)
%pip install azure-ai-ml azure-identity

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ------- -------------------------------- 2.6/13.2 MB 11.6 MB/s eta 0:00:01
   ----------------------------- ---------- 9.7/13.2 MB 23.2 MB/s eta 0:00:01
   ---------------------------------------- 13.2/13.2 MB 26.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 3.5/3.5 MB 70.0 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.1.1
    Uninstalling wrapt-2.1.1:
      Successfully uninstalled wrapt-2.1.1
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# 커널 작동확인 코드
print("bb")

bb


ENDPOINT_URL = "http://0be9559c-dbaa-467c-a59f-387551467a9f.koreacentral.azurecontainer.io/score"
API_KEY = "5T5y73pKpeH8itTa5T7K9OXtZH05RWKn"

In [35]:
import requests
import base64

# =============================================
# ★★★ 반드시 변경해야 하는 부분 ★★★
# =============================================
ENDPOINT_URL = "http://0be9559c-dbaa-467c-a59f-387551467a9f.koreacentral.azurecontainer.io/score"
API_KEY = "5T5y73pKpeH8itTa5T7K9OXtZH05RWKn"
# =============================================

DEBUG_MODE = False   # ← True로 바꾸면 payload 상세 로그 출력 (렉 발생 시 False 유지 추천)

def call_azure_endpoint(image_path, frame_id=0):
    """Azure ML 엔드포인트 호출 - 버퍼링 해결 버전"""
    with open(image_path, "rb") as f:
        image_data = f.read()
    
    base64_image = base64.b64encode(image_data).decode("utf-8")
    image_with_prefix = f"data:image/png;base64,{base64_image}"
    
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "Inputs": {
            "WebServiceInput0": [
                {
                    "image": image_with_prefix,
                    "id": frame_id,
                    "category": ""
                }
            ]
        }
    }
    
    # ★★★ 변경된 부분: DEBUG_MODE일 때만 출력 ★★★
    if DEBUG_MODE:
        print(f"\n[DEBUG] Frame {frame_id} payload 구조:")
        print(payload)
    
    response = requests.post(ENDPOINT_URL, headers=headers, json=payload)
    
    if response.status_code != 200:
        print("[DEBUG] === 서버 에러 응답 본문 ===")
        print(response.text)
        response.raise_for_status()
    
    result = response.json()
    output_item = result.get("Results", {}).get("WebServiceOutput0", [{}])[0] if "Results" in result else result.get("WebServiceOutput0", [{}])[0]
    
    label = output_item.get("Scored Labels", "Negative")
    conf = output_item.get("Scored Probabilities_fire", 0.0)
    
    return label, conf

print("✅ Cell 1 버퍼링 해결 버전 적용 완료")

✅ Cell 1 버퍼링 해결 버전 적용 완료


In [ ]:
import os
import glob
from collections import deque
from tqdm import tqdm   # ★★★ 추가된 부분: tqdm 진행바 ★★★

# =============================================
# Reliability Validation Layer (fire-only)
# =============================================
queue = deque(maxlen=10)
ewma_alpha = 0.4
min_conf = 0.65

def update_reliability(frame_label, confidence):
    if confidence < min_conf:
        queue.append(0)
        return "Green (정상)", 0.0
    
    is_fire = 1 if frame_label.strip().lower() == "fire" else 0
    queue.append(is_fire)
    
    fire_ratio = (sum(queue) / len(queue)) * 100
    ewma = 0.0
    for v in queue:
        ewma = ewma * (1 - ewma_alpha) + v * ewma_alpha
    
    if fire_ratio >= 75 or ewma >= 0.75:
        status = "Red (위험)"
    elif fire_ratio >= 40 or ewma >= 0.40:
        status = "Orange (경계)"
    else:
        status = "Green (정상)"
    
    return status, ewma

# =============================================
# ★★★ 테스트 이미지 폴더 ★★★
# =============================================
IMAGE_FOLDER = r"E:\newfolder\Egle_project\pytuning\bbfire30f"
# =============================================

image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, "*.jpg")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.png")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.jpeg")))

print(f"=== E-gle Eye 30장 자동 테스트 시작 ===\n총 이미지: {len(image_paths)}장\n")

# ★★★ 변경된 부분: tqdm 진행바 + 간결 출력 ★★★
for i, img_path in tqdm(enumerate(image_paths), total=len(image_paths), desc="테스트 진행"):
    filename = os.path.basename(img_path)
    label, conf = call_azure_endpoint(img_path, frame_id=i)
    status, ewma = update_reliability(label, conf)
    
    print(f"Frame {i+1:2d} | {filename:25} | 라벨:{label:8} | Fire확률:{conf:.4f} | "
          f"Fire비중:{sum(queue)/len(queue)*100:5.1f}% | EWMA:{ewma:.3f} | 상태:{status}")

print("\n=== 테스트 완료 ===")

=== E-gle Eye 30장 자동 테스트 시작 ===
총 이미지: 30장



테스트 진행:   3%|▎         | 1/30 [00:01<00:55,  1.91s/it]

Frame  1 | 01.png                    | 라벨:smoke    | Fire확률:0.0508 | Fire비중:  0.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행:   7%|▋         | 2/30 [00:04<00:56,  2.02s/it]

Frame  2 | 02.png                    | 라벨:smoke    | Fire확률:0.1711 | Fire비중:  0.0% | EWMA:0.000 | 상태:Green (정상)


...

Frame 28 | 28.png                    | 라벨:fire     | Fire확률:0.5563 | Fire비중: 85.7% | EWMA:0.000 | 상태:Green (정상)


테스트 진행:  97%|█████████▋| 29/30 [00:54<00:01,  1.77s/it]

Frame 29 | 29.png                    | 라벨:smoke    | Fire확률:0.4306 | Fire비중: 71.4% | EWMA:0.000 | 상태:Green (정상)


테스트 진행: 100%|██████████| 30/30 [00:56<00:00,  1.88s/it]

Frame 30 | 30.png                    | 라벨:smoke    | Fire확률:0.0508 | Fire비중: 57.1% | EWMA:0.000 | 상태:Green (정상)

=== 테스트 완료 ===


기존 시스템의 문제점

window_size=7 → 초기 smoke 구간에서 Fire비중 느리게 상승 (Frame 16 Green 유지)
Red 기준 75% → 화재 시작 후 너무 늦게 격상 (Frame 19에서야 Red)
화재 종료 후 과거 fire 프레임이 큐에 오래 남음 (Frame 22~27 Fire비중 100% 유지)
불 완전 전소 시 conf 낮아지는데도 Green으로 급격히 떨어지지 않음 (Frame 28~30)

In [22]:
import os
import glob
from collections import deque
from tqdm import tqdm

# =============================================
# Reliability Validation Layer (window_size=10 + 30장 피드백 완전 적용)
# =============================================
queue = deque(maxlen=10)          # ★★★ 피드백 반영: 7 → 10 ★★★
ewma_alpha = 0.42
min_conf = 0.55

def update_reliability(frame_label, confidence):
    if confidence < min_conf:
        queue.append(0)
        return "Green (정상)", 0.0
    
    is_fire = 1 if frame_label.strip().lower() == "fire" else 0
    queue.append(is_fire)
    
    fire_ratio = (sum(queue) / len(queue)) * 100
    ewma = 0.0
    for v in queue:
        ewma = ewma * (1 - ewma_alpha) + v * ewma_alpha
    
    if fire_ratio >= 60 or ewma >= 0.60:
        status = "Red (위험)"
    elif fire_ratio >= 32 or ewma >= 0.32:
        status = "Orange (경계)"
    else:
        status = "Green (정상)"
    
    return status, ewma

# =============================================
# ★★★ 테스트 이미지 폴더 ★★★
# =============================================
IMAGE_FOLDER = r"E:\newfolder\Egle_project\pytuning\bbfire30f"
# =============================================

image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, "*.jpg")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.png")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.jpeg")))

print(f"=== E-gle Eye 30장 자동 테스트 시작 (window_size=10 + 피드백 완전 적용) ===\n총 이미지: {len(image_paths)}장\n")

for i, img_path in tqdm(enumerate(image_paths), total=len(image_paths), desc="테스트 진행"):
    filename = os.path.basename(img_path)
    label, conf = call_azure_endpoint(img_path, frame_id=i)
    status, ewma = update_reliability(label, conf)
    
    print(f"Frame {i+1:2d} | {filename:25} | 라벨:{label:8} | Fire확률:{conf:.4f} | "
          f"Fire비중(최근10프레임):{sum(queue)/len(queue)*100:5.1f}% | EWMA:{ewma:.3f} | 상태:{status}")

print("\n=== 테스트 완료 ===")

=== E-gle Eye 30장 자동 테스트 시작 (window_size=10 + 피드백 완전 적용) ===
총 이미지: 30장



테스트 진행:   3%|▎         | 1/30 [00:01<00:50,  1.73s/it]

Frame  1 | 01.png                    | 라벨:smoke    | Fire확률:0.1961 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행:   7%|▋         | 2/30 [00:03<00:50,  1.79s/it]

Frame  2 | 02.png                    | 라벨:smoke    | Fire확률:0.0635 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)


...

Frame 28 | 28.png                    | 라벨:smoke    | Fire확률:0.3013 | Fire비중(최근10프레임): 80.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행:  97%|█████████▋| 29/30 [00:50<00:01,  1.70s/it]

Frame 29 | 29.png                    | 라벨:fire     | Fire확률:0.6949 | Fire비중(최근10프레임): 80.0% | EWMA:0.736 | 상태:Red (위험)


테스트 진행: 100%|██████████| 30/30 [00:52<00:00,  1.76s/it]

Frame 30 | 30.png                    | 라벨:smoke    | Fire확률:0.1961 | Fire비중(최근10프레임): 70.0% | EWMA:0.000 | 상태:Green (정상)

=== 테스트 완료 ===


In [24]:
import os
import glob
from collections import deque
from tqdm import tqdm

# =============================================
# Reliability Validation Layer (최종 버전)
# =============================================
queue = deque(maxlen=10)
ewma_alpha = 0.45
min_conf = 0.42   # ★★★ normal 인식 강화 위해 낮춤 ★★★

def update_reliability(frame_label, confidence):
    raw_label = frame_label.strip().lower()
    
    # ★★★ 라벨 매핑 강화 (normal 인식 해결 핵심) ★★★
    if raw_label == "fire":
        display_label = "fire"
    elif confidence < 0.25:                     # smoke인데 확률 매우 낮으면 normal로 판단
        display_label = "normal"
    else:
        display_label = "not_fire (smoke)"
    
    if confidence < min_conf:
        queue.append(0)
        return "Green (정상)", 0.0, display_label
    
    is_fire = 1 if frame_label.strip().lower() == "fire" else 0
    queue.append(is_fire)
    
    fire_ratio = (sum(queue) / len(queue)) * 100
    ewma = 0.0
    for v in queue:
        ewma = ewma * (1 - ewma_alpha) + v * ewma_alpha
    
    if fire_ratio >= 55 or ewma >= 0.55:
        status = "Red (위험)"
    elif fire_ratio >= 28 or ewma >= 0.28:
        status = "Orange (경계)"
    else:
        status = "Green (정상)"
    
    return status, ewma, display_label

# =============================================
# ★★★ 테스트 이미지 폴더 ★★★
# =============================================
IMAGE_FOLDER = r"E:\newfolder\Egle_project\pytuning\bbfire30f"
# =============================================

image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, "*.jpg")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.png")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.jpeg")))

print(f"=== E-gle Eye 30장 자동 테스트 시작 (window_size=10 + 라벨 표시 개선) ===\n총 이미지: {len(image_paths)}장\n")

for i, img_path in tqdm(enumerate(image_paths), total=len(image_paths), desc="테스트 진행"):
    filename = os.path.basename(img_path)
    label, conf = call_azure_endpoint(img_path, frame_id=i)
    status, ewma, display_label = update_reliability(label, conf)
    
    print(f"Frame {i+1:2d} | {filename:25} | 라벨:{display_label:15} | Fire확률:{conf:.4f} | "
          f"Fire비중(최근10프레임):{sum(queue)/len(queue)*100:5.1f}% | EWMA:{ewma:.3f} | 상태:{status}")

print("\n=== 테스트 완료 ===")

=== E-gle Eye 30장 자동 테스트 시작 (window_size=10 + 라벨 표시 개선) ===
총 이미지: 30장



테스트 진행:   3%|▎         | 1/30 [00:01<00:54,  1.89s/it]

Frame  1 | 01.png                    | 라벨:normal          | Fire확률:0.0508 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행:   7%|▋         | 2/30 [00:04<00:56,  2.02s/it]

Frame  2 | 02.png                    | 라벨:normal          | Fire확률:0.0635 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)


...

Frame 28 | 28.png                    | 라벨:not_fire (smoke) | Fire확률:0.3013 | Fire비중(최근10프레임): 90.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행:  97%|█████████▋| 29/30 [00:58<00:01,  1.93s/it]

Frame 29 | 29.png                    | 라벨:not_fire (smoke) | Fire확률:0.4306 | Fire비중(최근10프레임): 80.0% | EWMA:0.300 | 상태:Red (위험)


테스트 진행: 100%|██████████| 30/30 [01:00<00:00,  2.01s/it]

Frame 30 | 30.png                    | 라벨:normal          | Fire확률:0.1961 | Fire비중(최근10프레임): 70.0% | EWMA:0.000 | 상태:Green (정상)

=== 테스트 완료 ===


지하철 5호선 화재 영상 이미지 프레임 테스트

In [32]:
import os
import glob
from collections import deque
from tqdm import tqdm

# =============================================
# Reliability Validation Layer (최종 버전)
# =============================================
queue = deque(maxlen=10)
ewma_alpha = 0.45
min_conf = 0.42   # ★★★ normal 인식 강화 위해 낮춤 ★★★

def update_reliability(frame_label, confidence):
    raw_label = frame_label.strip().lower()
    
    # ★★★ 라벨 매핑 강화 (normal 인식 해결 핵심) ★★★
    if raw_label == "fire":
        display_label = "fire"
    elif confidence < 0.25:                     # smoke인데 확률 매우 낮으면 normal로 판단
        display_label = "normal"
    else:
        display_label = "not_fire (smoke)"
    
    if confidence < min_conf:
        queue.append(0)
        return "Green (정상)", 0.0, display_label
    
    is_fire = 1 if frame_label.strip().lower() == "fire" else 0
    queue.append(is_fire)
    
    fire_ratio = (sum(queue) / len(queue)) * 100
    ewma = 0.0
    for v in queue:
        ewma = ewma * (1 - ewma_alpha) + v * ewma_alpha
    
    if fire_ratio >= 55 or ewma >= 0.55:
        status = "Red (위험)"
    elif fire_ratio >= 28 or ewma >= 0.28:
        status = "Orange (경계)"
    else:
        status = "Green (정상)"
    
    return status, ewma, display_label

# =============================================
# ★★★ 테스트 이미지 폴더 ★★★
# =============================================
IMAGE_FOLDER = r"E:\newfolder\Egle_project\pytuning\subway_frame30f"
# =============================================

image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, "*.jpg")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.png")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.jpeg")))

print(f"=== E-gle Eye 30장 자동 테스트 시작 (window_size=10 + 라벨 표시 개선) ===\n총 이미지: {len(image_paths)}장\n")

for i, img_path in tqdm(enumerate(image_paths), total=len(image_paths), desc="테스트 진행"):
    filename = os.path.basename(img_path)
    label, conf = call_azure_endpoint(img_path, frame_id=i)
    status, ewma, display_label = update_reliability(label, conf)
    
    print(f"Frame {i+1:2d} | {filename:25} | 라벨:{display_label:15} | Fire확률:{conf:.4f} | "
          f"Fire비중(최근10프레임):{sum(queue)/len(queue)*100:5.1f}% | EWMA:{ewma:.3f} | 상태:{status}")

print("\n=== 테스트 완료 ===")

=== E-gle Eye 30장 자동 테스트 시작 (window_size=10 + 라벨 표시 개선) ===
총 이미지: 30장



테스트 진행:   3%|▎         | 1/30 [00:02<00:58,  2.01s/it]

Frame  1 | 01.png                    | 라벨:normal          | Fire확률:0.0060 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행:   7%|▋         | 2/30 [00:04<01:03,  2.26s/it]

Frame  2 | 02.png                    | 라벨:normal          | Fire확률:0.0708 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)


...

Frame 28 | 28.png                    | 라벨:fire            | Fire확률:0.9321 | Fire비중(최근10프레임): 40.0% | EWMA:0.589 | 상태:Red (위험)


테스트 진행:  97%|█████████▋| 29/30 [00:59<00:02,  2.15s/it]

Frame 29 | 29.png                    | 라벨:not_fire (smoke) | Fire확률:0.3499 | Fire비중(최근10프레임): 40.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행: 100%|██████████| 30/30 [01:01<00:00,  2.03s/it]

Frame 30 | 30.png                    | 라벨:fire            | Fire확률:0.9479 | Fire비중(최근10프레임): 50.0% | EWMA:0.628 | 상태:Red (위험)

=== 테스트 완료 ===


# 화재 진압 포말 방사기의 위력 영상 이미지 프레임 테스트

In [33]:
import os
import glob
from collections import deque
from tqdm import tqdm

# =============================================
# Reliability Validation Layer (최종 버전)
# =============================================
queue = deque(maxlen=10)
ewma_alpha = 0.45
min_conf = 0.42   # ★★★ normal 인식 강화 위해 낮춤 ★★★

def update_reliability(frame_label, confidence):
    raw_label = frame_label.strip().lower()
    
    # ★★★ 라벨 매핑 강화 (normal 인식 해결 핵심) ★★★
    if raw_label == "fire":
        display_label = "fire"
    elif confidence < 0.25:                     # smoke인데 확률 매우 낮으면 normal로 판단
        display_label = "normal"
    else:
        display_label = "not_fire (smoke)"
    
    if confidence < min_conf:
        queue.append(0)
        return "Green (정상)", 0.0, display_label
    
    is_fire = 1 if frame_label.strip().lower() == "fire" else 0
    queue.append(is_fire)
    
    fire_ratio = (sum(queue) / len(queue)) * 100
    ewma = 0.0
    for v in queue:
        ewma = ewma * (1 - ewma_alpha) + v * ewma_alpha
    
    if fire_ratio >= 55 or ewma >= 0.55:
        status = "Red (위험)"
    elif fire_ratio >= 28 or ewma >= 0.28:
        status = "Orange (경계)"
    else:
        status = "Green (정상)"
    
    return status, ewma, display_label

# =============================================
# ★★★ 테스트 이미지 폴더 ★★★
# =============================================
IMAGE_FOLDER = r"E:\newfolder\Egle_project\pytuning\fire_frames"
# =============================================

image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, "*.jpg")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.png")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.jpeg")))

print(f"=== E-gle Eye 30장 자동 테스트 시작 (window_size=10 + 라벨 표시 개선) ===\n총 이미지: {len(image_paths)}장\n")

for i, img_path in tqdm(enumerate(image_paths), total=len(image_paths), desc="테스트 진행"):
    filename = os.path.basename(img_path)
    label, conf = call_azure_endpoint(img_path, frame_id=i)
    status, ewma, display_label = update_reliability(label, conf)
    
    print(f"Frame {i+1:2d} | {filename:25} | 라벨:{display_label:15} | Fire확률:{conf:.4f} | "
          f"Fire비중(최근10프레임):{sum(queue)/len(queue)*100:5.1f}% | EWMA:{ewma:.3f} | 상태:{status}")

print("\n=== 테스트 완료 ===")

=== E-gle Eye 30장 자동 테스트 시작 (window_size=10 + 라벨 표시 개선) ===
총 이미지: 33장



테스트 진행:   3%|▎         | 1/33 [00:02<01:04,  2.01s/it]

Frame  1 | frame-45.png              | 라벨:normal          | Fire확률:0.1623 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행:   6%|▌         | 2/33 [00:03<00:59,  1.91s/it]

Frame  2 | frame-46.png              | 라벨:normal          | Fire확률:0.0400 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)


...

Frame 31 | frame-91.png              | 라벨:normal          | Fire확률:0.0113 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행:  97%|█████████▋| 32/33 [01:04<00:01,  1.98s/it]

Frame 32 | frame-93.png              | 라벨:normal          | Fire확률:0.0325 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행: 100%|██████████| 33/33 [01:06<00:00,  2.02s/it]

Frame 33 | frame-95.png              | 라벨:normal          | Fire확률:0.0113 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)

=== 테스트 완료 ===


# E-gle Eye 로직 업데이트 버전

버전: window_size=10 + 라벨 표시 개선 + Smoke 양성 처리 강화

목적: 초기 연기 미탐지 + 잔여 연기 후 상태 급락 문제 완전 해결

In [37]:
import os
import glob
from collections import deque
from tqdm import tqdm

# =============================================
# Reliability Validation Layer (v2.0 - 원준님 승인 버전)
# =============================================
queue = deque(maxlen=10)
ewma_alpha = 0.38          # ★★★ 로직 담당 변경: 0.45 → 0.38 (안정성 강화) ★★★
min_conf = 0.42
ORANGE_THRESHOLD = 42      # ★★★ 로직 담당 변경: 28 → 42 (원 spec 복구) ★★★
RED_THRESHOLD = 72         # ★★★ 로직 담당 변경: 55 → 72 (원 spec 복구) ★★★
downgrade_counter = 0      # ★★★ 로직 담당 추가: Red→Orange 다운그레이드 지연용 ★★★

def update_reliability(frame_label, confidence):
    global downgrade_counter
    raw_label = frame_label.strip().lower()
    
    # ★★★ 라벨 매핑 강화 + Smoke 가중치 추가 (로직 담당 핵심 개선) ★★★
    if raw_label == "fire":
        display_label = "fire"
        is_fire_weight = 1.0
    elif raw_label in ["not_fire (smoke)", "smoke"] and confidence >= 0.30:
        display_label = "not_fire (smoke)"
        is_fire_weight = 0.4          # ★★★ smoke 가중치 신규 추가 ★★★
    elif confidence < 0.25:
        display_label = "normal"
        is_fire_weight = 0.0
    else:
        display_label = "not_fire (smoke)"
        is_fire_weight = 0.0
    
    # ★★★ min_conf 처리 개선 (hysteresis 적용) ★★★
    if confidence < min_conf:
        current_fire_ratio = (sum(queue) / len(queue) * 100) if queue else 0
        if current_fire_ratio < ORANGE_THRESHOLD:   # 낮은 상태일 때만 강제 Green
            queue.append(0)
            downgrade_counter = 0
            return "Green (정상)", 0.0, display_label
        else:
            # Red/Orange 유지 중이면 min_conf 무시하고 정상 계산 진행
            is_fire_weight = 0.0
    
    # Fire 카운트 (가중치 적용)
    queue.append(is_fire_weight)
    
    fire_ratio = (sum(queue) / len(queue)) * 100
    ewma = 0.0
    for v in queue:
        ewma = ewma * (1 - ewma_alpha) + v * ewma_alpha
    
    # ★★★ 상태 판단 (threshold + 다운그레이드 지연 로직) ★★★
    if fire_ratio >= RED_THRESHOLD or ewma >= 0.72:
        status = "Red (위험)"
        downgrade_counter = 0
    elif fire_ratio >= ORANGE_THRESHOLD or ewma >= 0.42:
        status = "Orange (경계)"
        downgrade_counter = 0
    else:
        downgrade_counter += 1
        if downgrade_counter >= 4:          # ★★★ Red→Orange 다운그레이드 최소 4프레임 지연 ★★★
            status = "Green (정상)"
            downgrade_counter = 0
        else:
            status = "Orange (경계)" if ewma >= 0.28 else "Red (위험)"  # 이전 상태 유지
    
    return status, ewma, display_label

# =============================================
# ★★★ 테스트 이미지 폴더 ★★★
# =============================================
IMAGE_FOLDER = r"E:\newfolder\Egle_project\pytuning\subway_frame30f"

image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, "*.jpg")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.png")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.jpeg")))

print(f"=== E-gle Eye v2.0 테스트 시작 (hysteresis + smoke 가중치 적용) ===\n총 이미지: {len(image_paths)}장\n")

for i, img_path in tqdm(enumerate(image_paths), total=len(image_paths), desc="테스트 진행"):
    filename = os.path.basename(img_path)
    label, conf = call_azure_endpoint(img_path, frame_id=i)   # 기존 함수 그대로 사용
    status, ewma, display_label = update_reliability(label, conf)
    
    print(f"Frame {i+1:2d} | {filename:25} | 라벨:{display_label:15} | Fire확률:{conf:.4f} | "
          f"Fire비중:{sum(queue)/len(queue)*100:5.1f}% | EWMA:{ewma:.3f} | 상태:{status}")

print("\n=== v2.0 테스트 완료 ===")

=== E-gle Eye v2.0 테스트 시작 (hysteresis + smoke 가중치 적용) ===
총 이미지: 30장



테스트 진행:   3%|▎         | 1/30 [00:02<01:05,  2.26s/it]

Frame  1 | 01.png                    | 라벨:normal          | Fire확률:0.0585 | Fire비중:  0.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행:   7%|▋         | 2/30 [00:04<01:00,  2.16s/it]

Frame  2 | 02.png                    | 라벨:normal          | Fire확률:0.0708 | Fire비중:  0.0% | EWMA:0.000 | 상태:Green (정상)


...

Frame 28 | 28.png                    | 라벨:fire            | Fire확률:0.9321 | Fire비중: 54.0% | EWMA:0.642 | 상태:Orange (경계)


테스트 진행:  97%|█████████▋| 29/30 [01:00<00:01,  1.81s/it]

Frame 29 | 29.png                    | 라벨:not_fire (smoke) | Fire확률:0.3499 | Fire비중: 54.0% | EWMA:0.398 | 상태:Orange (경계)


테스트 진행: 100%|██████████| 30/30 [01:02<00:00,  2.08s/it]

Frame 30 | 30.png                    | 라벨:fire            | Fire확률:0.9479 | Fire비중: 64.0% | EWMA:0.627 | 상태:Orange (경계)

=== v2.0 테스트 완료 ===


# 발생한 이벤트를 기록할 수 있게 만든 코드

In [31]:
import os
import glob
from collections import deque
from tqdm import tqdm

# =============================================
# Reliability Validation Layer (TypeError 완전 해결 버전)
# =============================================
queue = deque(maxlen=10)
ewma_alpha = 0.45
min_conf = 0.42

events = []                     # ★★★ 이벤트 기록 리스트 ★★★
prev_status = "Green (정상)"    # ★★★ 이전 상태 비교용 ★★★

def update_reliability(frame_label, confidence, frame_id):
    global prev_status, events   # ★★★ global 선언을 최상단으로 이동 ★★★
    
    raw_label = frame_label.strip().lower()
    
    if raw_label == "fire":
        display_label = "fire"
    elif confidence < 0.25:
        display_label = "normal"
    else:
        display_label = "not_fire (smoke)"
    
    if confidence < min_conf:
        queue.append(0)
        status = "Green (정상)"
        ewma = 0.0
    else:
        is_fire = 1 if raw_label == "fire" else 0
        queue.append(is_fire)
        
        fire_ratio = (sum(queue) / len(queue)) * 100
        ewma = 0.0
        for v in queue:
            ewma = ewma * (1 - ewma_alpha) + v * ewma_alpha
        
        if fire_ratio >= 58 or ewma >= 0.58:
            status = "Red (위험)"
        elif fire_ratio >= 28 or ewma >= 0.28:
            status = "Orange (경계)"
        else:
            status = "Green (정상)"
    
    # ★★★ 이벤트 감지 및 강조 출력 ★★★
    if status != prev_status:
        event_msg = f"[EVENT] Frame {frame_id+1} | {prev_status} → {status} | 라벨:{display_label}"
        events.append(event_msg)
        print(f"\n{'='*70}\n{event_msg}\n{'='*70}")
    
    prev_status = status
    return status, ewma, display_label

# =============================================
# ★★★ 테스트 이미지 폴더 ★★★
# =============================================
IMAGE_FOLDER = r"E:\newfolder\Egle_project\pytuning\bbfire30f"
# =============================================

image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, "*.jpg")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.png")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.jpeg")))

print(f"=== E-gle Eye 30장 자동 테스트 시작 (이벤트 별도 출력 + TypeError 해결) ===\n총 이미지: {len(image_paths)}장\n")

for i, img_path in tqdm(enumerate(image_paths), total=len(image_paths), desc="테스트 진행"):
    filename = os.path.basename(img_path)
    label, conf = call_azure_endpoint(img_path, frame_id=i)
    status, ewma, display_label = update_reliability(label, conf, i)   # ★★★ frame_id 전달 (TypeError 해결 핵심) ★★★
    
    print(f"Frame {i+1:2d} | {filename:25} | 라벨:{display_label:15} | Fire확률:{conf:.4f} | "
          f"Fire비중(최근10프레임):{sum(queue)/len(queue)*100:5.1f}% | EWMA:{ewma:.3f} | 상태:{status}")

# ★★★ 이벤트 요약 출력 ★★★
print("\n" + "="*70)
print("=== 특정 이벤트 요약 ===")
if events:
    for ev in events:
        print(ev)
else:
    print("발생한 이벤트가 없습니다.")
print("="*70)

print("\n=== 테스트 완료 ===")

=== E-gle Eye 30장 자동 테스트 시작 (이벤트 별도 출력 + TypeError 해결) ===
총 이미지: 30장



테스트 진행:   3%|▎         | 1/30 [00:02<01:19,  2.72s/it]

Frame  1 | 01.png                    | 라벨:normal          | Fire확률:0.1961 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행:   7%|▋         | 2/30 [00:04<01:06,  2.39s/it]

Frame  2 | 02.png                    | 라벨:normal          | Fire확률:0.1711 | Fire비중(최근10프레임):  0.0% | EWMA:0.000 | 상태:Green (정상)


...


[EVENT] Frame 28 | Red (위험) → Green (정상) | 라벨:not_fire (smoke)
Frame 28 | 28.png                    | 라벨:not_fire (smoke) | Fire확률:0.3013 | Fire비중(최근10프레임): 80.0% | EWMA:0.000 | 상태:Green (정상)


테스트 진행:  97%|█████████▋| 29/30 [00:57<00:02,  2.02s/it]


[EVENT] Frame 29 | Green (정상) → Red (위험) | 라벨:fire
Frame 29 | 29.png                    | 라벨:fire            | Fire확률:0.6949 | Fire비중(최근10프레임): 80.0% | EWMA:0.738 | 상태:Red (위험)


테스트 진행: 100%|██████████| 30/30 [01:00<00:00,  2.00s/it]


[EVENT] Frame 30 | Red (위험) → Green (정상) | 라벨:normal
Frame 30 | 30.png                    | 라벨:normal          | Fire확률:0.0508 | Fire비중(최근10프레임): 70.0% | EWMA:0.000 | 상태:Green (정상)

=== 특정 이벤트 요약 ===
[EVENT] Frame 16 | Green (정상) → Orange (경계) | 라벨:fire
[EVENT] Frame 17 | Orange (경계) → Red (위험) | 라벨:fire
[EVENT] Frame 28 | Red (위험) → Green (정상) | 라벨:not_fire (smoke)
[EVENT] Frame 29 | Green (정상) → Red (위험) | 라벨:fire
[EVENT] Frame 30 | Red (위험) → Green (정상) | 라벨:normal

=== 테스트 완료 ===
